# Logistic regression — Boomerang vs Sticky Boomerang

Two experiments: binary (8 features, 2 true signals) and multinomial (6 features, 3 classes, dense coefficients).

**What to expect:**

- **Binary, plain Boomerang:** means track the sklearn MLE, stds match Laplace approximation. All P(=0) ≈ 0.
- **Binary, Sticky:** signals recovered, null coefs have P(=0) > 0.5.
- **Multiclass:** posterior means should correlate strongly with sklearn's fitted coefficients. Sticky with uniform small κ shrinks all coefficients — useful as a check that the sampler is working, less interesting as a sparsity story since the truth is dense.

**Warning sign:** if posterior stds ≈ `prior_std` everywhere, the sampler isn't exploring the likelihood.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

import os
os.chdir('../..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.glm import make_logistic_regression, make_kappa_vector_glm
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky

## Helpers

In [ ]:
def run_both(target, kappa, N_skel=20_000, N_resample=50_000, burnin=0.1):
    """Run plain and sticky Boomerang on the same target, return resampled posteriors."""
    sampler = AutomaticBoomerangSampler(
        grad_target=target.grad_target, D=target.D, refresh_rate=1.0, thinning='pli',
    )
    sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

    sampler_s = StickyAutomaticBoomerangSampler(
        grad_target=target.grad_target, D=target.D, refresh_rate=1.0,
        kappa=kappa, thinning='pli',
    )
    sampler_s.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

    r = sampler.sample(N=N_skel, diagnostics=False)
    rs = sampler_s.sample(N=N_skel, diagnostics=False)

    x_ref_np = target.x_ref.cpu().numpy()
    samples = resample_pdmp_path(
        r['positions'].cpu().numpy(), r['velocities'].cpu().numpy(),
        r['times'].cpu().numpy(), x_ref_np,
        N_resample=N_resample, burnin_frac=burnin,
    )
    samples_s = resample_pdmp_path_sticky(
        rs['positions'].cpu().numpy(), rs['velocities'].cpu().numpy(),
        rs['times'].cpu().numpy(), x_ref_np,
        N_resample=N_resample, burnin_frac=burnin,
    )
    return samples, samples_s


def calibration_plot(samples, samples_s, true_coefs, ref_coefs, labels,
                     is_signal=None, title=''):
    """Coefficient-wise posterior intervals vs truth and a point-estimate reference."""
    means = samples.mean(0); stds = samples.std(0)
    means_s = samples_s.mean(0); stds_s = samples_s.std(0)
    idx = np.arange(len(true_coefs))
    off = 0.18

    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(idx) + 3), 4))
    ax.errorbar(idx - off, means, yerr=2 * stds, fmt='o', color='C0',
                label='Boomerang ±2σ', capsize=3, markersize=5)
    ax.errorbar(idx + off, means_s, yerr=2 * stds_s, fmt='s', color='C1',
                label='Sticky ±2σ', capsize=3, markersize=5)
    ax.scatter(idx, true_coefs, marker='x', color='k', s=70, linewidths=2,
               label='true', zorder=5)
    if ref_coefs is not None:
        ax.scatter(idx, ref_coefs, marker='_', color='grey', s=120, linewidths=2,
                   label='sklearn MLE', zorder=4)
    if is_signal is not None:
        for i in np.where(is_signal)[0]:
            ax.axvspan(i - 0.45, i + 0.45, color='gold', alpha=0.15)
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_xticks(idx)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('coefficient value')
    ax.set_title(title)
    ax.legend(loc='best', frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()

## 1. Binary logistic regression

True signals only in β₂ and β₅.

In [ ]:
rng = np.random.default_rng(0)
N, D = 300, 8
X = rng.normal(size=(N, D))
beta_true = np.zeros(D); beta_true[[1, 4]] = [2.0, -1.5]
intercept_true = -0.3
logits = X @ beta_true + intercept_true
y = rng.binomial(1, 1 / (1 + np.exp(-logits)))
X = (X - X.mean(0)) / X.std(0)

true_coefs = np.concatenate([[intercept_true], beta_true])
is_signal = true_coefs != 0

# sklearn MLE reference (no regularisation to match flat-prior intuition)
clf = LogisticRegression(penalty=None, max_iter=2000).fit(X, y)
mle_coefs = np.concatenate([clf.intercept_, clf.coef_.ravel()])

print(f"Binary: N={N}, D={D}, class balance={y.mean():.2f}")

In [ ]:
import pymc as pm

X_np = np.asarray(X)
y_np = np.asarray(y).astype(int)

with pm.Model() as logreg_bin_model:
    intercept = pm.Normal('intercept', mu=0.0, sigma=10.0)
    betas = pm.Normal('betas', mu=0.0, sigma=1.0, shape=D)
    logits = intercept + X_np @ betas
    pm.Bernoulli('y', logit_p=logits, observed=y_np)

    nuts_bin = pm.sample(
        draws=2000, tune=1000, chains=2,
        target_accept=0.9, progressbar=True, random_seed=0,
    )

nuts_int = nuts_bin.posterior['intercept'].values.reshape(-1)
nuts_bet = nuts_bin.posterior['betas'].values.reshape(-1, D)
samples_nuts = np.column_stack([nuts_int, nuts_bet])
print(f"NUTS (binary): {samples_nuts.shape}")

In [ ]:
target = make_logistic_regression(
    torch.tensor(X, dtype=torch.float64),
    torch.tensor(y, dtype=torch.long),
    prior_std=1.0, intercept_prior_std=10.0,
)

kappa = torch.full((target.D,), 1.0, dtype=torch.float64)
kappa[0] = 1e6  # intercept never sticks

samples, samples_s = run_both(target, kappa, N_skel=20_000)

In [ ]:
beta_true

In [ ]:
means = samples.mean(0); stds = samples.std(0)
means_s = samples_s.mean(0); stds_s = samples_s.std(0)
means_n = samples_nuts.mean(0); stds_n = samples_nuts.std(0)
p_zero = (np.abs(samples_s) < 1e-8).mean(0)

header = (f"{'coef':<10} {'true':>7} {'MLE':>7}"
          f"  |  {'NUTS μ':>7} {'NUTS σ':>7}"
          f"  |  {'Boom μ':>7} {'Boom σ':>7}"
          f"  |  {'Stky μ':>7} {'Stky σ':>7} {'P(=0)':>6}")
print(header); print('-' * len(header))
for i in range(len(true_coefs)):
    lbl = 'intercept' if i == 0 else f'β_{i}'
    star = ' *' if is_signal[i] else ''
    print(f"{lbl:<10} {true_coefs[i]:>7.3f} {mle_coefs[i]:>7.3f}"
          f"  |  {means_n[i]:>7.3f} {stds_n[i]:>7.3f}"
          f"  |  {means[i]:>7.3f} {stds[i]:>7.3f}"
          f"  |  {means_s[i]:>7.3f} {stds_s[i]:>7.3f} {p_zero[i]:>6.2f}{star}")

print(f"\nStd ratio (sampler/NUTS, averaged):")
print(f"  Boomerang: {(stds / stds_n).mean():.2f}  (want ≈ 1.0)"
      f"   Sticky (signals): {(stds_s[is_signal] / stds_n[is_signal]).mean():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(true_coefs))
offset = 0.22

ax.errorbar(idx - offset, means_n, yerr=2 * stds_n, fmt='D', color='C2',
            label='NUTS ±2σ', capsize=3, markersize=5)
ax.errorbar(idx, means, yerr=2 * stds, fmt='o', color='C0',
            label='Boomerang ±2σ', capsize=3, markersize=5)
ax.errorbar(idx + offset, means_s, yerr=2 * stds_s, fmt='s', color='C1',
            label='Sticky ±2σ', capsize=3, markersize=5)
ax.scatter(idx, true_coefs, marker='x', color='k', s=70, linewidths=2,
           label='true', zorder=5)

for i in np.where(is_signal)[0]:
    ax.axvspan(i - 0.45, i + 0.45, color='gold', alpha=0.15)

ax.axhline(0, color='grey', lw=0.5)
ax.set_xticks(idx)
ax.set_xticklabels(['int.'] + [f'β_{i}' for i in range(1, len(true_coefs))])
ax.set_ylabel('coefficient value')
ax.set_title('Binary logistic — NUTS (green) is the reference')
ax.legend(loc='best', frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
j = np.argmax(np.abs(true_coefs))

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, samp, name, color in [
    (axes[0], samples_nuts, 'NUTS', 'C2'),
    (axes[1], samples, 'Boomerang', 'C0'),
    (axes[2], samples_s, 'Sticky', 'C1'),
]:
    ax.plot(samp[:, j], lw=0.4, color=color)
    ax.axhline(true_coefs[j], color='k', lw=1, ls='--',
               label=f'true = {true_coefs[j]:.2f}')
    ax.set_title(f'{name}: β_{j} trace')
    ax.set_xlabel('sample index')
    ax.legend(loc='best', fontsize=8, frameon=False)
axes[0].set_ylabel(f'β_{j}')
plt.tight_layout(); plt.show()


In [ ]:
j = np.argmin(np.abs(true_coefs))

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, samp, name, color in [
    (axes[0], samples_nuts[:100, ], 'NUTS', 'C2'),
    (axes[1], samples[:100, ], 'Boomerang', 'C0'),
    (axes[2], samples_s[:100, ], 'Sticky', 'C1'),
]:
    ax.plot(samp[:, j], lw=0.4, color=color)
    ax.axhline(true_coefs[j], color='k', lw=1, ls='--',
               label=f'true = {true_coefs[j]:.2f}')
    ax.set_title(f'{name}: β_{j} trace')
    ax.set_xlabel('sample index')
    ax.legend(loc='best', fontsize=8, frameon=False)
axes[0].set_ylabel(f'β_{j}')
plt.tight_layout(); plt.show()